<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_08-3_Memory_%EA%B4%80%EB%A6%AC%EC%99%80_Checkpoint_%EB%B3%B5%EA%B5%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 지능시스템 08-3 | Memory 관리와 Checkpoint 복구

## 학습 목표
1. **Working Context**, **Session Summary**, **Long-term Memory**를 구분한다.
2. 작업 기록을 **Compaction(요약)** 하여 필요한 정보만 남긴다.
3. 장기 기억을 JSON 파일에 저장하고 **다음 세션에서 다시 불러온다**.
4. 실행 상태를 **Checkpoint**에 저장하고 중단된 작업을 재개한다.
5. **Checkpoint는 작업 재개를 위한 상태**, **Long-term Memory는 재사용하는 정보**라는 차이를 직접 확인한다.

모든 데이터는 가상의 수업 정책이며 실제 사용자 개인정보를 다루지 않습니다.

결과는 실행 중인 Colab 세션의 `/content/agent_memory_lab/`(로컬 Jupyter에서는 현재 폴더 아래)에 만들어집니다. Colab 런타임이 초기화되면 로컬 파일이 사라질 수 있습니다.

## 1. 환경 준비: 저장 폴더와 예제 데이터
외부 라이브러리를 설치하지 않습니다. Python 기본 라이브러리인 `json`, `pathlib`, `copy`를 사용합니다.

In [1]:
# json: 파이썬의 딕셔너리와 리스트를 JSON 파일로 저장하고 다시 읽습니다.
import json
# Path: 파일과 폴더의 경로를 다루기 편리하게 합니다.
from pathlib import Path
# deepcopy: 원본 데이터를 건드리지 않고 복사할 때 사용합니다.
from copy import deepcopy

# Colab 환경이면 /content 아래에, 일반 Jupyter면 현재 폴더 아래에 저장합니다.
BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
DATA_DIR = BASE_DIR / "agent_memory_lab"
DATA_DIR.mkdir(parents=True, exist_ok=True)  # 폴더가 이미 있어도 오류가 나지 않습니다.

MEMORY_FILE = DATA_DIR / "long_term_memory.json"   # 재사용할 장기 지식
CHECKPOINT_FILE = DATA_DIR / "agent_checkpoint.json" # 중단된 실행을 재개할 상태

print("실습 데이터 폴더:", DATA_DIR)
print("장기 기억 파일:", MEMORY_FILE.name)
print("체크포인트 파일:", CHECKPOINT_FILE.name)

실습 데이터 폴더: /content/agent_memory_lab
장기 기억 파일: long_term_memory.json
체크포인트 파일: agent_checkpoint.json


## 2. 세 종류의 정보를 구분하기

| 구분 | 담는 정보 | 예시 | 유지 범위(이 실습) |
|---|---|---|---|
| Working Context | 바로 다음 판단에 필요한 내용 | 방금 읽은 정책 문장 | 현재 실행 중 메모리 |
| Session Summary | 긴 작업의 핵심 진행 상황 | 확인한 정책·남은 할 일 | 현재 세션에서 활용 |
| Long-term Memory | 다음 세션에서도 재사용할 **확인된** 정보 | 평가 규정의 출처를 포함한 사실 | JSON 파일로 지속 |

**중요:** Session Summary와 Long-term Memory가 모두 `Memory`라는 이름을 갖더라도 보존 기간과 목적은 다릅니다. 또한 세 층은 이해를 돕기 위한 설계 예시이지 모든 제품의 표준 구조는 아닙니다.

### 2-1. Working Context와 Session Summary 구현

`observe()`는 새 결과를 작업 공간에 넣습니다. `compact()`는 핵심 요약을 별도로 보관하고 최근 기록만 남깁니다. 실제 LLM 요약 대신 학생이 확인 가능한 문자열을 사용합니다.

In [2]:
class AgentMemory:
    """현재 작업 기록과 세션 요약, 장기 기억을 분리해 보관하는 단순한 클래스입니다."""

    def __init__(self):
        # working: 실행 중 바로 참고할 관찰 기록(일시적 데이터)
        self.working = []
        # session_summary: 긴 기록 대신 남기는 핵심 요약
        self.session_summary = []
        # long_term: 작업이 끝나도 재사용할 지식 (나중에 파일로 저장)
        self.long_term = {}

    def observe(self, item):
        """Tool 결과처럼 이번 실행에서 얻은 정보를 추가합니다."""
        self.working.append(item)

    def compact(self, summary, keep_last=1):
        """핵심 정보를 요약으로 남기고, 오래된 작업 기록을 줄입니다."""
        # 음수가 전달되면 실수일 수 있으므로 미리 막습니다.
        if not isinstance(keep_last, int) or keep_last < 0:
            raise ValueError("keep_last는 0 이상의 정수여야 합니다.")
        self.session_summary.append(summary)
        # keep_last=0일 때 [-0:]은 전체를 뜻하므로 별도 처리합니다.
        self.working = self.working[-keep_last:] if keep_last > 0 else []

    def remember(self, key, value):
        """향후 재사용할 검증된 정보를 장기 기억에 추가합니다."""
        self.long_term[key] = value

    def recall(self, key):
        """해당 정보가 있으면 반환하고, 없으면 None을 반환합니다."""
        return self.long_term.get(key)

memory = AgentMemory()
print("초기 Working Context:", memory.working)
print("초기 Session Summary:", memory.session_summary)
print("초기 Long-term Memory:", memory.long_term)

초기 Working Context: []
초기 Session Summary: []
초기 Long-term Memory: {}


### 2-2. 관찰 결과를 쌓고 Compaction 적용하기
강의 정책을 읽는 상황을 가정합니다. 세부 검색 기록은 줄이더라도 핵심 결정은 세션 요약에 남겨야 합니다.

In [3]:
# 세 번의 Tool 실행으로 서로 다른 정보가 들어왔다고 가정합니다.
memory.observe({"tool": "course_search", "text": "과제 평가 기준이 포함된 문서 발견"})
memory.observe({"tool": "course_read", "text": "예제 정책: 과제 20%, 출석 20%"})
memory.observe({"tool": "course_search", "text": "AI 활용 규정 확인이 아직 필요함"})

print("[요약 전] Working Context 항목 수:", len(memory.working))
for index, item in enumerate(memory.working, start=1):
    print(index, item["text"])

# 이미 확인한 중요한 내용만 요약합니다. 실제 서비스라면 요약의 사실성 검증이 필요합니다.
memory.compact(
    summary="예제 정책에서 과제 20%, 출석 20%를 확인함. AI 활용 규정은 미확인.",
    keep_last=1  # 최근 기록 한 개만 Working Context에 보관
)

print("\n[요약 후] Working Context 항목 수:", len(memory.working))
print("남은 Working Context:", memory.working)
print("Session Summary:", memory.session_summary)

[요약 전] Working Context 항목 수: 3
1 과제 평가 기준이 포함된 문서 발견
2 예제 정책: 과제 20%, 출석 20%
3 AI 활용 규정 확인이 아직 필요함

[요약 후] Working Context 항목 수: 1
남은 Working Context: [{'tool': 'course_search', 'text': 'AI 활용 규정 확인이 아직 필요함'}]
Session Summary: ['예제 정책에서 과제 20%, 출석 20%를 확인함. AI 활용 규정은 미확인.']


## 3. Long-term Memory: 다음 세션에 쓸 정보 저장

장기 기억에는 **재사용 가치가 있고 확인된 정보만** 저장합니다. 여기서는 예제 규정과 출처를 함께 저장합니다. 실제 수업 운영에서는 확인되지 않은 추정이나 개인정보를 자동 저장하면 안 됩니다.

In [4]:
# 사실과 출처를 함께 저장하면 나중에 검증하기 편합니다.
memory.remember(
    "course_policy_example",
    {
        "fact": "예제 정책: 과제 20%, 출석 20%",
        "source": "실습용 가상 강의 정책",
        "verified": True,
    },
)

# 확인되지 않은 내용을 장기 기억에 쓰지 않는 간단한 검사 함수입니다.
def remember_if_verified(agent_memory, key, record):
    """verified=True인 딕셔너리만 장기 기억으로 저장합니다."""
    if not isinstance(record, dict) or record.get("verified") is not True:
        return False  # 저장 거부
    agent_memory.remember(key, record)
    return True

saved = remember_if_verified(
    memory,
    "unconfirmed_rule",
    {"fact": "AI 활용은 무조건 허용된다", "verified": False},
)
print("미확인 정보 저장 성공 여부:", saved)  # False가 정상입니다.
print("장기 기억에 등록한 키:", list(memory.long_term))

미확인 정보 저장 성공 여부: False
장기 기억에 등록한 키: ['course_policy_example']


### 3-1. JSON 파일에 쓰고 다시 읽기

**중요:** 파이썬 객체는 런타임이 종료되면 사라집니다. JSON 파일에 기록해야 같은 저장 공간에 접근하는 다음 실행에서 다시 읽을 수 있습니다.

In [5]:
def save_long_term(agent_memory, path):
    """AgentMemory의 long_term 딕셔너리를 JSON 파일로 저장합니다."""
    # ensure_ascii=False: 한글을 \uXXXX가 아니라 한글 그대로 저장합니다.
    # indent=2: 사람이 읽기 좋게 줄을 나눕니다.
    path.write_text(
        json.dumps(agent_memory.long_term, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def load_long_term(agent_memory, path):
    """파일이 있으면 장기 기억을 불러옵니다."""
    if not path.exists():
        return False  # 아직 저장한 파일이 없다는 뜻
    data = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise ValueError("장기 기억 JSON은 딕셔너리여야 합니다.")
    agent_memory.long_term = data
    return True

save_long_term(memory, MEMORY_FILE)
print("장기 기억 저장 완료:", MEMORY_FILE)

# 새 객체를 만들면 working과 세션 요약은 비어 있습니다.
new_session_memory = AgentMemory()
print("새 세션의 Working Context:", new_session_memory.working)
print("장기 기억 불러오기 성공:", load_long_term(new_session_memory, MEMORY_FILE))
print("다시 읽은 정보:", new_session_memory.recall("course_policy_example"))

장기 기억 저장 완료: /content/agent_memory_lab/long_term_memory.json
새 세션의 Working Context: []
장기 기억 불러오기 성공: True
다시 읽은 정보: {'fact': '예제 정책: 과제 20%, 출석 20%', 'source': '실습용 가상 강의 정책', 'verified': True}


## 4. Checkpoint: 중단된 작업을 다시 시작하기

Checkpoint는 **같은 작업(run)을 재개하는 데 필요한 실행 상태**를 저장합니다. 장기 기억과 달리 현재 단계(`step`), 완료 목록, 대기 목록 등을 가집니다.

이번 실습의 작업: **3개 항목의 정책을 순서대로 확인하기**

- `grading_policy`: 평가 규정
- `attendance_policy`: 출석 규정
- `ai_policy`: AI 활용 규정

각 항목은 실제 웹이나 파일을 읽지 않고 미리 만든 가상 데이터에서 확인합니다.

In [6]:
# 외부 API 없이 재현 가능한 가상 정책 데이터입니다.
POLICY_DB = {
    "grading_policy": "예제: 과제 20%, 중간 30%, 기말 30%",
    "attendance_policy": "예제: 출석 20%",
    "ai_policy": "예제: AI 사용 시 활용 사실을 표시한다",
}

# 한 번의 실행(run)에 필요한 상태입니다.
def initial_state():
    return {
        "run_id": "run-001",               # 작업 식별자
        "goal": "강의 정책 3개 확인",       # 작업 목표
        "step": 0,                       # 완료된 단계 수
        "completed": [],                 # 완료한 작업 목록
        "pending": list(POLICY_DB),       # 앞으로 처리할 작업 목록
        "results": {},                    # 이번 작업에서 확인한 결과
    }

state = initial_state()
print(json.dumps(state, ensure_ascii=False, indent=2))

{
  "run_id": "run-001",
  "goal": "강의 정책 3개 확인",
  "step": 0,
  "completed": [],
  "pending": [
    "grading_policy",
    "attendance_policy",
    "ai_policy"
  ],
  "results": {}
}


### 4-1. 한 단계 실행하는 함수
**중요:** 완료한 작업은 `pending` 목록에서 제거하므로 복구할 때 중복 처리할 가능성을 줄입니다. 실제 외부 시스템에서는 중복 요청 방지를 위한 별도 장치(멱등성 키 등)도 필요합니다.

In [7]:
def run_one_step(state):
    """남아 있는 정책 항목 중 하나를 확인하고 실행 상태를 변경합니다."""
    if not state["pending"]:
        return False  # 남은 작업이 없음 = 종료

    # 가장 앞의 대기 작업을 가져옵니다. 여기서는 순서를 고정합니다.
    task_name = state["pending"].pop(0)
    # 간단한 'Tool' 역할: 준비된 가상 정책 사전에서 값을 읽습니다.
    result = POLICY_DB[task_name]

    # Tool 실행 결과와 진행 상태를 모두 기록합니다.
    state["results"][task_name] = result
    state["completed"].append(task_name)
    state["step"] += 1
    print(f"[Step {state['step']}] {task_name} 완료 → {result}")
    return True

# 일부러 첫 번째 단계만 실행하고 중단합니다.
run_one_step(state)
print("현재 완료:", state["completed"])
print("앞으로 할 일:", state["pending"])

[Step 1] grading_policy 완료 → 예제: 과제 20%, 중간 30%, 기말 30%
현재 완료: ['grading_policy']
앞으로 할 일: ['attendance_policy', 'ai_policy']


### 4-2. Checkpoint 저장·불러오기
JSON 파일로 저장합니다. 이해를 돕기 위해 간단한 형식만 사용하며, 실제 서비스에는 파일 손상 검증·동시성 제어 등이 추가로 필요합니다.

In [8]:
def save_checkpoint(current_state, path):
    """중단 시점의 실행 상태 전체를 JSON 파일로 저장합니다."""
    path.write_text(
        json.dumps(current_state, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def load_checkpoint(path):
    """저장된 체크포인트가 있으면 상태 딕셔너리를 반환합니다."""
    if not path.exists():
        return None
    data = json.loads(path.read_text(encoding="utf-8"))
    required_keys = {"run_id", "goal", "step", "completed", "pending", "results"}
    if not isinstance(data, dict) or not required_keys.issubset(data):
        raise ValueError("체크포인트에 필수 정보가 없습니다.")
    return data

save_checkpoint(state, CHECKPOINT_FILE)
print("Checkpoint 저장 완료:", CHECKPOINT_FILE)

# 프로그램을 다시 시작한 상황을 흉내 냅니다.
state = None
print("실행 중 상태를 비웠습니다:", state)
restored_state = load_checkpoint(CHECKPOINT_FILE)
print("복구된 단계:", restored_state["step"])
print("복구된 대기 작업:", restored_state["pending"])

Checkpoint 저장 완료: /content/agent_memory_lab/agent_checkpoint.json
실행 중 상태를 비웠습니다: None
복구된 단계: 1
복구된 대기 작업: ['attendance_policy', 'ai_policy']


### 4-3. 중단 지점부터 재개하기
복구된 상태에서 남은 항목만 처리합니다. 매 단계가 끝날 때마다 Checkpoint를 갱신합니다.

In [9]:
# 복구한 상태를 현재 실행 상태로 사용합니다.
state = restored_state

while state["pending"]:
    run_one_step(state)
    # 작업이 한 단계 끝날 때마다 상태를 파일로 기록합니다.
    save_checkpoint(state, CHECKPOINT_FILE)

print("\n최종 완료 목록:", state["completed"])
print("최종 단계:", state["step"])
print("남은 작업:", state["pending"])
print("최종 검색 결과:")
for name, result in state["results"].items():
    print("-", name, ":", result)

[Step 2] attendance_policy 완료 → 예제: 출석 20%
[Step 3] ai_policy 완료 → 예제: AI 사용 시 활용 사실을 표시한다

최종 완료 목록: ['grading_policy', 'attendance_policy', 'ai_policy']
최종 단계: 3
남은 작업: []
최종 검색 결과:
- grading_policy : 예제: 과제 20%, 중간 30%, 기말 30%
- attendance_policy : 예제: 출석 20%
- ai_policy : 예제: AI 사용 시 활용 사실을 표시한다


## 5. Checkpoint와 Memory를 비교해 보기

- Checkpoint: `run-001`을 몇 단계까지 수행했는지 보여줍니다.
- Long-term Memory: 예제 수업 정책의 확인된 정보를 다시 읽을 수 있게 합니다.

**Checkpoint의 완료된 실행 상태를 장기 기억이라고 부르지 않습니다.** 서로 다른 역할을 가진 데이터입니다.

In [10]:
# 장기 기억은 실행 단계와 관계없이 다른 세션에서 읽을 수 있습니다.
long_term = new_session_memory.recall("course_policy_example")
# Checkpoint는 바로 그 실행(run-001)이 어디까지 갔는지를 기록합니다.
checkpoint = load_checkpoint(CHECKPOINT_FILE)

print("[Long-term Memory] 다시 쓸 수 있는 사실:", long_term["fact"])
print("[Checkpoint] 실행 ID:", checkpoint["run_id"])
print("[Checkpoint] 완료 단계:", checkpoint["step"])
print("[Checkpoint] 작업 완료 여부:", not checkpoint["pending"])

[Long-term Memory] 다시 쓸 수 있는 사실: 예제 정책: 과제 20%, 출석 20%
[Checkpoint] 실행 ID: run-001
[Checkpoint] 완료 단계: 3
[Checkpoint] 작업 완료 여부: True


## 6. 자동 테스트: 코드가 올바르게 동작했는지 확인하기

`assert`는 조건이 참이면 그냥 넘어가고, 거짓이면 오류를 발생시킵니다. 따라서 학생들이 결과를 눈으로만 확인하지 않고 코드로 검증할 수 있습니다.

In [11]:
# 테스트 1: Compaction 후 최근 기록만 남았는가?
assert len(memory.working) == 1
assert len(memory.session_summary) == 1

# 테스트 2: 검증되지 않은 정보가 저장되지 않았는가?
assert memory.recall("unconfirmed_rule") is None

# 테스트 3: 다른 객체에서도 장기 기억을 읽을 수 있는가?
assert new_session_memory.recall("course_policy_example") == memory.recall("course_policy_example")

# 테스트 4: Checkpoint로 중단된 작업을 재개해 모두 완료했는가?
assert state["step"] == len(POLICY_DB)
assert len(state["completed"]) == len(POLICY_DB)
assert state["pending"] == []

# 테스트 5: 작업이 중복 실행되지 않았는가?
assert len(state["completed"]) == len(set(state["completed"]))

# 테스트 6: 파일을 다시 읽어도 최종 단계가 같은가?
assert load_checkpoint(CHECKPOINT_FILE)["step"] == len(POLICY_DB)

print("✅ 자동 테스트 6개 모두 통과!")

✅ 자동 테스트 6개 모두 통과!


**핵심 정리:** *Compaction은 현재 Context를 줄이고, Long-term Memory는 재사용할 정보를 저장하며, Checkpoint는 실행 중단 지점에서 작업을 재개하도록 돕습니다.*